# 🗳️ Lesson 05: Voting & Stacking

**Time:** ~2.5 hours  **Prerequisites:** Lessons 01–04

So far every ensemble was built from *one family* of model: all trees. But Lesson 01's two ingredients were **accurate** and **diverse**,
and nothing is more diverse than a linear model sitting next to a gradient-boosted forest. Today we combine models of *different kinds*:

* **Voting / averaging**: a fixed, simple rule (everyone gets an equal say, or fixed weights).
* **Stacking**: a *learned* rule: a **meta-model** discovers how much to trust each base model.

### 🎯 Learning goals
1. Distinguish **hard** and **soft** voting; compute both by hand.
2. Measure **diversity** between models (correlation of errors) and connect it to ensemble gains.
3. Build **stacking from scratch** with **out-of-fold (OOF) predictions**: and demonstrate the data-leakage disaster you get without them.
4. Use scikit-learn's `StackingRegressor` / `StackingClassifier`, and choose a sensible meta-learner (non-negative weights!).
5. Know when stacking is worth its complexity.

> 💡 **Analogy.** Voting is a jury where every juror has one equal vote. Stacking is a **judge** who has watched each juror's track record on past cases and now
> weighs their opinions accordingly: "Priya is great with money crimes (I'll weight her heavily there; Sam is usually wrong) I'll listen less."

In [1]:
import sys; sys.path.append("../utils")          # lets us import the course helpers
from course_utils import *                       # load_housing(), check(), set_style(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style()
np.random.seed(SEED)
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier, StackingClassifier,
                              RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, StackingRegressor)
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import seaborn as sns
import xgboost as xgb

## 1 · Voting classifiers

### The dataset: will this borrower default?
We switch to our **classification** dataset: 5,000 small loans, ~14.5% defaulted. With such **imbalance**, *accuracy is a trap*: a model that says "nobody defaults" scores ~85.5% and is useless.
We'll judge with **ROC-AUC**: the probability that a randomly chosen defaulter gets a higher risk score than a randomly chosen non-defaulter (0.5 = coin flip, 1.0 = perfect).

In [2]:
loans = load_loans()
print(f"{len(loans)} loans, default rate = {loans['defaulted'].mean():.1%}   |   accuracy of 'nobody defaults' = {1 - loans['defaulted'].mean():.1%}")

yl = loans["defaulted"]
Xl = loans.drop(columns=["defaulted", "customer_id"])                      # customer_id is an identifier, not a feature!
Xl_train, Xl_test, yl_train, yl_test = train_test_split(Xl, yl, test_size=0.25, stratify=yl, random_state=SEED)

prep_l = make_preprocessor(Xl_train, cat_cols=["region", "loan_purpose"], scale=True)    # impute + scale + one-hot
Al_train, Al_test = prep_l.fit_transform(Xl_train), prep_l.transform(Xl_test)
print("feature matrix:", Al_train.shape)

5000 loans, default rate = 14.5%   |   accuracy of 'nobody defaults' = 85.5%
feature matrix: (3750, 23)


We'll use four deliberately *different* classifiers: a linear model (smooth, global), k-nearest-neighbours (local, distance-based), a random forest (bagged trees) and histogram gradient boosting (boosted trees).

In [3]:
cls_models = {
    "Logistic":     LogisticRegression(max_iter=2000),
    "KNN":          KNeighborsClassifier(n_neighbors=25),
    "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1),
    "HistGB":       HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3, random_state=SEED),
}
Z_test_cls = pd.DataFrame({name: m.fit(Al_train, yl_train).predict_proba(Al_test)[:, 1] for name, m in cls_models.items()})   # P(default)

rows = [{"model": n, "ROC-AUC": roc_auc_score(yl_test, Z_test_cls[n]),
         "accuracy @0.5": accuracy_score(yl_test, Z_test_cls[n] > 0.5), "F1 @0.5": f1_score(yl_test, Z_test_cls[n] > 0.5)} for n in cls_models]
pd.DataFrame(rows).round(3)

,model,ROC-AUC,accuracy @0.5,F1 @0.5
0,Logistic,0.745,0.858,0.227
1,KNN,0.732,0.855,0.108
2,RandomForest,0.751,0.858,0.183
3,HistGB,0.747,0.859,0.267


Four weak-ish models (AUC ≈ 0.73–0.75; this problem has a lot of irreducible noise). Notice accuracy ≈ 85% for all (the trap in action) and a low F1 at the naive 0.5 threshold: with a 14% base rate, almost no one reaches a 50% default probability. *(Choosing thresholds is a business decision; for ranking quality we use AUC.)*

### Hard vs. soft voting

* **Hard voting:** each model casts a *class* vote; the majority wins.
* **Soft voting:** average the models' **predicted probabilities**, then threshold. Confident models count for more.

### 🧪 Worked Example 5.1: one applicant, three models

| model A | model B | model C |
|---|---|---|
| P(default) = 0.45 | 0.45 | 0.90 |

* **Hard:** votes are *no, no, yes* → majority says **no default**.
* **Soft:** mean = (0.45 + 0.45 + 0.90)/3 = **0.60** → says **default**.

Same models, opposite decisions! Model C is *very* sure, A and B are barely on the fence; soft voting hears that, hard voting doesn't.
This is why **soft voting is usually better** (when models output sensible probabilities), and why it also gives you a probability for ROC-AUC.

In [4]:
hard = VotingClassifier(list(cls_models.items()), voting="hard").fit(Al_train, yl_train)
soft = VotingClassifier(list(cls_models.items()), voting="soft").fit(Al_train, yl_train)

print(f"hard voting : accuracy {accuracy_score(yl_test, hard.predict(Al_test)):.3f}   F1 {f1_score(yl_test, hard.predict(Al_test)):.3f}   (no probabilities -> no AUC)")
p_soft = soft.predict_proba(Al_test)[:, 1]
print(f"soft voting : accuracy {accuracy_score(yl_test, p_soft > 0.5):.3f}   F1 {f1_score(yl_test, p_soft > 0.5):.3f}   ROC-AUC {roc_auc_score(yl_test, p_soft):.4f}")
print(f"best single model's AUC: {max(roc_auc_score(yl_test, Z_test_cls[n]) for n in cls_models):.4f}")

hard voting : accuracy 0.859   F1 0.162   (no probabilities -> no AUC)
soft voting : accuracy 0.856   F1 0.174   ROC-AUC 0.7606
best single model's AUC: 0.7508


The soft-voting committee has a **higher AUC than any member**. Why? Time to check Lesson 01's diversity condition: how correlated are the models' predictions?

## 2 · Diversity, measured

To judge diversity fairly we'll use **out-of-fold (OOF) predictions** on the training set: for each row, the prediction of a model that was trained on *other* folds (5-fold CV).
These are honest predictions (the model never saw that row) and we'll need exactly this object for stacking in a minute.

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
Z_oof_cls = pd.DataFrame({name: cross_val_predict(m, Al_train, yl_train, cv=skf, method="predict_proba")[:, 1]
                          for name, m in cls_models.items()})                         # shape: (n_train, n_models)

fig, ax = plt.subplots(figsize=(5.8, 4.4))
sns.heatmap(Z_oof_cls.corr(), annot=True, fmt=".2f", cmap="Blues", vmin=0.6, vmax=1, ax=ax, cbar=False)
ax.set_title("Correlation of OOF predicted probabilities"); plt.tight_layout(); plt.show()

Reading the heatmap: the **two tree ensembles are the most alike** (≈ 0.92), the **KNN is the odd one out** (≈ 0.77–0.81 with everyone). Diversity is highest where model *families* differ: consistent with the recipe.

## 3 · Stacking: learn the weights

**Stacking** (Wolpert, 1992) trains a second-level model: the **meta-learner**: whose *inputs are the predictions of the base models* and whose target is the true label.

```
   training data ──► base model 1 ──► prediction p1 ─┐
                 ├─► base model 2 ──► prediction p2 ─┼──►  META-LEARNER  ──►  final prediction
                 └─► base model 3 ──► prediction p3 ─┘      (e.g. logistic / non-negative linear regression)
```

### ⚠️ The golden rule of stacking: the meta-learner must be trained on **out-of-fold** predictions

What if we simply ask each base model to predict the *training* set (the data it was fit on) and feed those predictions to the meta-learner?
A random forest has effectively **memorised** its training data: its in-sample predictions are nearly perfect. The meta-learner sees "the forest is always right!" and gives it enormous weight. On new data the forest is merely good, and the stack collapses.

That's **leakage**, and we can demonstrate it. We'll build both versions *from scratch*:

In [6]:
def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))

# --- NAIVE (wrong): base models predict the data they were trained on ---
Z_naive = pd.DataFrame({n: m.predict_proba(Al_train)[:, 1] for n, m in cls_models.items()})     # models already fitted on Al_train above
meta_naive = LogisticRegression(max_iter=2000).fit(logit(Z_naive), yl_train)

# --- CORRECT: meta-learner trained on out-of-fold predictions (Z_oof_cls from the previous section) ---
meta_oof = LogisticRegression(max_iter=2000).fit(logit(Z_oof_cls), yl_train)

# at test time both use the SAME base-model predictions on the test set
auc_naive = roc_auc_score(yl_test, meta_naive.predict_proba(logit(Z_test_cls))[:, 1])
auc_oof   = roc_auc_score(yl_test, meta_oof.predict_proba(logit(Z_test_cls))[:, 1])

weights = pd.DataFrame({"naive stack": meta_naive.coef_[0], "OOF stack": meta_oof.coef_[0]}, index=cls_models)
print(weights.round(2))
print(f"\ntest AUC  naive stack: {auc_naive:.4f}   |   OOF stack: {auc_oof:.4f}   |   plain soft vote: {roc_auc_score(yl_test, p_soft):.4f}")

              naive stack  OOF stack
Logistic            -1.50       0.46
KNN                 -0.07      -0.01
RandomForest         7.53       0.57
HistGB              -1.30       0.09

test AUC  naive stack: 0.7230   |   OOF stack: 0.7576   |   plain soft vote: 0.7606


### 🔎 The leakage, caught in the act
* **Naive stack:** the meta-learner gave the **random forest a giant weight (≈ 7.5)** and nearly ignored the rest, because on the training set the forest looks flawless. Test AUC *falls below every single base model*.
* **OOF stack:** sensible weights spread across logistic regression and the forest; test AUC is close to (here slightly below) the plain soft vote.

> ⚠️ **Never let a meta-learner see predictions that a base model made on its own training rows.** That is the single most common stacking bug.

### The complete stacking recipe
1. Choose a CV split (say 5-fold).
2. For each base model, produce **OOF predictions** for the whole training set (`cross_val_predict`).
3. Train the **meta-learner** on those OOF predictions (features) and the true target.
4. **Refit** every base model on the *full* training set; their predictions on new data become the meta-learner's inputs.

scikit-learn does all four steps in `StackingClassifier` / `StackingRegressor`.

In [7]:
stack_cls = StackingClassifier(estimators=list(cls_models.items()), final_estimator=LogisticRegression(max_iter=2000),
                               cv=skf, stack_method="predict_proba", n_jobs=1).fit(Al_train, yl_train)
print(f"scikit-learn StackingClassifier test AUC: {roc_auc_score(yl_test, stack_cls.predict_proba(Al_test)[:, 1]):.4f}")

scikit-learn StackingClassifier test AUC: 0.7605


Here the stack and the plain average are neck-and-neck: with only four models, similar skill, and a noisy target, *learning weights* has little to add over *equal weights*. Stacking pays when the base models differ **a lot in quality** or in **where** they are good. Our regression problem is exactly that.

## 4 · Stacking on the housing data (where it really shines)

We'll build four regressors that differ in family *and* in how they see the target:

| model | personality |
|---|---|
| **LogLinear** | Ridge regression on `log(price)`, exponentiated back: handles the multiplicative structure of prices |
| **RandomForest** | bagged trees on raw price |
| **HistGB** | histogram gradient boosting |
| **XGBoost** | gradient boosting with subsampling |

In [8]:
X_train, X_test, y_train, y_test = get_housing_split()
prep_h = make_preprocessor(X_train, scale=True)
Ah_train, Ah_test = prep_h.fit_transform(X_train), prep_h.transform(X_test)
yh_train, yh_test = y_train.values, y_test.values
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

reg_models = {
    "LogLinear":    TransformedTargetRegressor(Ridge(alpha=1.0), func=np.log, inverse_func=np.exp),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_features=0.4, random_state=SEED, n_jobs=-1),
    "HistGB":       HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, max_depth=4, random_state=SEED),
    "XGBoost":      xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=SEED),
}
kf_reg = KFold(n_splits=5, shuffle=True, random_state=SEED)

Z_oof_reg  = pd.DataFrame({n: cross_val_predict(m, Ah_train, yh_train, cv=kf_reg) for n, m in reg_models.items()})   # honest train predictions
Z_test_reg = pd.DataFrame({n: m.fit(Ah_train, yh_train).predict(Ah_test) for n, m in reg_models.items()})            # models refit on ALL train

single = pd.Series({n: rmse(yh_test, Z_test_reg[n]) for n in reg_models}, name="test RMSE")
single.round(1)

LogLinear       108.1
RandomForest    115.8
HistGB          110.4
XGBoost         107.8
Name: test RMSE, dtype: float64

All four models land in the 108–116 range. Now check the diversity of their **errors** (OOF residuals): the quantity that Lesson 01's formula cares about:

In [9]:
resid_corr = Z_oof_reg.sub(yh_train, axis=0).corr()
fig, ax = plt.subplots(figsize=(5.8, 4.4))
sns.heatmap(resid_corr, annot=True, fmt=".2f", cmap="Blues", vmin=0.8, vmax=1, ax=ax, cbar=False)
ax.set_title("Correlation of OOF *errors*"); plt.tight_layout(); plt.show()

The three tree-based models make nearly the same mistakes (≈ 0.98 correlated) but the log-linear model's errors are noticeably different (≈ 0.92). That's our diversity. Now combine them three ways:

1. **Equal-weight average** (simple voting).
2. **Stacking with OLS** (unconstrained linear meta-learner).
3. **Stacking with non-negative weights** (`LinearRegression(positive=True)`).

Why non-negative? A negative weight means "predict the *opposite* of this model": a symptom of a meta-learner exploiting noise among highly correlated inputs. Forcing weights ≥ 0 makes them behave like a **weighted average**: stable, interpretable, and far less prone to overfit.

In [10]:
results = {"Equal-weight average": rmse(yh_test, Z_test_reg.mean(axis=1))}
learned = {}
for label, meta in [("Stack: OLS meta", LinearRegression()), ("Stack: non-negative meta", LinearRegression(positive=True))]:
    meta.fit(Z_oof_reg, yh_train)                                  # trained on OUT-OF-FOLD predictions
    results[label] = rmse(yh_test, meta.predict(Z_test_reg))
    learned[label] = np.append(meta.coef_, meta.intercept_)

comparison = pd.concat([single, pd.Series(results, name="test RMSE")]).sort_values()
print(comparison.round(1).to_string())
print()
print(pd.DataFrame(learned, index=list(reg_models) + ["intercept"]).round(3))

Stack: non-negative meta     96.4
Stack: OLS meta              97.0
Equal-weight average         97.1
XGBoost                     107.8
LogLinear                   108.1
HistGB                      110.4
RandomForest                115.8

              Stack: OLS meta  Stack: non-negative meta
LogLinear               0.389                     0.384
RandomForest           -0.092                     0.000
HistGB                  0.079                     0.048
XGBoost                 0.607                     0.562
intercept               7.928                     3.262


### 🔎 What happened?

* The **equal-weight average** of four models (RMSE ≈ 97) beats *every single member* by roughly 10%. That's the Lesson 01 formula paying out: errors partly cancel.
* The **non-negative stack** squeezes out a little more (≈ 96) by learning that the log-linear model and XGBoost deserve most of the weight, and the random forest almost none (its errors are covered by the other trees).
* **OLS** stacking assigns a *negative* weight to the random forest (textbook overfitting-by-flexibility) and gains nothing over simple averaging.

Remember the dilution problem from Lesson 01 (equal-weight averaging with a weak linear model *hurt* the best member)? Here the linear model is *not* weak, because we gave it the right target transform (log). **Modelling the target correctly can matter more than the choice of algorithm.**

### Let scikit-learn do it
`StackingRegressor` runs the same recipe; it should reproduce our non-negative stack closely:

In [11]:
stack_reg = StackingRegressor(estimators=list(reg_models.items()), final_estimator=LinearRegression(positive=True), cv=kf_reg)
stack_reg.fit(Ah_train, yh_train)
print(f"StackingRegressor test RMSE: {rmse(yh_test, stack_reg.predict(Ah_test)):.1f}")
print("meta-learner weights:", {n: round(float(w), 3) for n, w in zip(reg_models, stack_reg.final_estimator_.coef_)})

StackingRegressor test RMSE: 96.4
meta-learner weights: {'LogLinear': 0.384, 'RandomForest': 0.0, 'HistGB': 0.048, 'XGBoost': 0.562}


## 5 · Variations, pitfalls and advice

### Blending (the cheap cousin)
Instead of K-fold OOF predictions, hold out a single **blend set** (say 20% of the training data): train base models on the rest, predict the blend set, fit the meta-learner there. Simpler and faster, but wastes data and gives a noisier meta-learner. Fine for huge datasets or quick experiments.

### `passthrough=True`
Feed the meta-learner the *original features too*, not just the base predictions. Powerful but dangerous: the meta-learner can overfit like any model. Use a regularised or tree-based meta-learner if you try it.

### Pitfall checklist ✅

| pitfall | cure |
|---|---|
| Meta-learner trained on in-sample predictions | **always OOF** (scikit-learn's `Stacking*` classes do this for you) |
| Too flexible a meta-learner | linear / ridge / non-negative linear; stacks of stacks rarely pay |
| Base models all from the same family | add *different* families (linear, trees, kNN, NN) and *different views* (log target, different features) |
| Judging the stack on the data used to build it | keep an untouched test set; nest CV for rigorous estimates (Lesson 06) |
| Latency & maintenance | a stack of 6 models is 6 things to deploy, monitor and retrain: is the 1–3% gain worth it? |
| Duplicated preprocessing leakage | put preprocessing *inside* a `Pipeline` per base model so each CV fold fits it only on its training part |

### When is stacking worth it?
* ✅ Competitions, or high-value predictions where 1–2% accuracy is worth real money.
* ✅ When you have strong but *different* models (log-linear vs. trees was a perfect example).
* ❌ When one model is far better than the rest (stacking can't conjure signal that isn't there).
* ❌ When interpretability and simple deployment dominate: a single tuned GBM plus SHAP (Lesson 08) is often the better product.

### ✋ Check your understanding

Try to answer *out loud* before you open the answer. If you can't say it simply, you don't own it yet.

**Q1. Three models give P(default) = 0.40, 0.40, 0.95. What do hard and soft voting (threshold 0.5) predict?**

<details><summary>Show answer</summary>

Hard: votes are no, no, yes → **no default**. Soft: mean = 0.583 → **default**. Soft voting lets the very confident model outweigh two fence-sitters.

</details>

**Q2. Why must the meta-learner be trained on *out-of-fold* predictions?**

<details><summary>Show answer</summary>

In-sample predictions of a flexible model (random forest, boosted trees) are nearly perfect because the model has memorised those rows. The meta-learner would over-trust that model; on new data its predictions are merely good, so the stack underperforms. OOF predictions mimic what the base models will look like on *unseen* data.

</details>

**Q3. Why constrain the meta-learner's weights to be non-negative?**

<details><summary>Show answer</summary>

Base-model predictions are highly correlated, so an unconstrained linear fit can use large positive and negative weights to chase noise. Non-negative weights force a weighted-average interpretation, which is stable, interpretable and generalises better.

</details>

**Q4. Your four models' OOF errors have pairwise correlations of 0.99. Is stacking likely to help? What would you do?**

<details><summary>Show answer</summary>

Very little: near-identical errors mean there's almost nothing to cancel (Lesson 01's ρ ≈ 1). Add models that differ in *family* (linear, kNN, neural net), in *target transform*, or in *feature view* before stacking.

</details>

## 6 · Problems

Variables from the lesson: `Z_oof_cls`, `Z_test_cls`, `yl_train`, `yl_test` (loans); `Z_oof_reg`, `Z_test_reg`, `Ah_train`, `Ah_test`, `yh_train`, `yh_test`, `kf_reg`, `rmse` (housing).

### 🧩 Problem 05.1 ⭐: Hard vs. soft, by hand

Four applicants, three models. Each row of `P` is `[model A, model B, model C]` = P(default):

```python
P = np.array([[0.45, 0.45, 0.90],
              [0.20, 0.60, 0.80],
              [0.40, 0.10, 0.30],
              [0.55, 0.49, 0.45]])
```

Using threshold 0.5, compute with NumPy: `hard`: the **majority-vote** class (0/1) for each applicant, and `soft`: the class (0/1) from the **mean probability**.
Which applicants get different decisions?

<details><summary>💡 Hint 1</summary>

Votes: `votes = (P > 0.5)` gives a boolean per model. The majority has more than half the votes: `votes.sum(axis=1) >= 2`.

</details>

<details><summary>💡 Hint 2</summary>

Soft: `P.mean(axis=1) > 0.5`.

</details>

<details><summary>💡 Hint 3</summary>

Convert booleans to ints with `.astype(int)`. Expect `hard = [0, 1, 0, 0]` and `soft = [1, 1, 0, 0]`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
P = np.array([[0.45, 0.45, 0.90],
              [0.20, 0.60, 0.80],
              [0.40, 0.10, 0.30],
              [0.55, 0.49, 0.45]])

hard = TODO   # YOUR CODE HERE
soft = TODO   # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("05.1", lambda: list(hard) == [0, 1, 0, 0] and list(soft) == [1, 1, 0, 0],
      "Expected hard = [0, 1, 0, 0] and soft = [1, 1, 0, 0].", needs=[hard, soft])

### 🧩 Problem 05.2 ⭐⭐: Who is the odd one out?

Using the OOF probabilities `Z_oof_cls` (columns: Logistic, KNN, RandomForest, HistGB):

1. Find the **most correlated pair** of models and the **least correlated pair**. Store them as sorted tuples in `most_pair` and `least_pair`, e.g. `("HistGB", "RandomForest")`.
2. Compute the **test AUC of an equal-weight soft vote of just the least-correlated pair** → `auc_diverse_pair`, and of the most-correlated pair → `auc_similar_pair` (use `Z_test_cls`).

*Reflection:* does the diverse pair beat the similar pair? Is that what Lesson 01 predicts? (Be careful; the *accuracy* of the members matters too.)

<details><summary>💡 Hint 1</summary>

`C = Z_oof_cls.corr()` is a symmetric matrix. Loop over pairs with `itertools.combinations(C.columns, 2)` and record `C.loc[a, b]`.

</details>

<details><summary>💡 Hint 2</summary>

`most_pair = max(pairs, key=...)`, `least_pair = min(pairs, key=...)`. Sort each pair: `tuple(sorted(pair))`.

</details>

<details><summary>💡 Hint 3</summary>

AUC of a pair: `roc_auc_score(yl_test, Z_test_cls[list(pair)].mean(axis=1))`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
from itertools import combinations

most_pair = TODO          # YOUR CODE HERE
least_pair = TODO         # YOUR CODE HERE
auc_diverse_pair = TODO   # YOUR CODE HERE
auc_similar_pair = TODO   # YOUR CODE HERE

# ---- self-check (don't edit) ----
def _expected():
    from itertools import combinations
    C = Z_oof_cls.corr()
    pc = {p: C.loc[p[0], p[1]] for p in combinations(C.columns, 2)}
    return tuple(sorted(max(pc, key=pc.get))), tuple(sorted(min(pc, key=pc.get)))
check("05.2", lambda: (tuple(most_pair), tuple(least_pair)) == _expected()
      and 0.7 < auc_diverse_pair < 0.8 and 0.7 < auc_similar_pair < 0.8,
      "Most correlated should be RandomForest & HistGB (≈ 0.92); compute pair AUCs from the TEST probabilities.",
      needs=[most_pair, least_pair, auc_diverse_pair, auc_similar_pair])

### 🧩 Problem 05.3 ⭐⭐: Choosing the meta-learner

Using the OOF predictions `Z_oof_reg` (train) and `Z_test_reg` (test) and the targets `yh_train`, `yh_test`:

* `rmse_equal`: test RMSE of the **equal-weight average** of the four models
* fit `LinearRegression(positive=True)` on `(Z_oof_reg, yh_train)`; store its test RMSE in `rmse_nnls` and its learned weights (`.coef_`) in `weights`

*Reflection:* which model did the meta-learner like best? Which did it effectively drop, and why do you think so?

<details><summary>💡 Hint 1</summary>

Equal weights: `Z_test_reg.mean(axis=1)` then `rmse(yh_test, ...)`.

</details>

<details><summary>💡 Hint 2</summary>

`meta = LinearRegression(positive=True).fit(Z_oof_reg, yh_train)`; predict with `meta.predict(Z_test_reg)`.

</details>

<details><summary>💡 Hint 3</summary>

`weights = meta.coef_`: one non-negative number per model, in the column order of `Z_oof_reg`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
rmse_equal = TODO   # YOUR CODE HERE
rmse_nnls = TODO    # YOUR CODE HERE
weights = TODO      # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("05.3", lambda: abs(rmse_equal - rmse(yh_test, Z_test_reg.mean(axis=1))) < 1e-6 and np.all(np.asarray(weights) >= -1e-9)
      and rmse_nnls < rmse_equal + 1.0 and rmse_nnls < 105,
      "Weights must be non-negative; the stack should be about as good as or better than the plain average (both ≈ 96-97).",
      needs=[rmse_equal, rmse_nnls, weights])

### 🧩 Problem 05.4 ⭐⭐⭐: Add a fifth model - generating OOF predictions yourself

Add an `ExtraTreesRegressor(n_estimators=300, max_features=0.5, random_state=SEED, n_jobs=-1)` to the committee.

1. Write `oof_predictions(model, X, y, cv)` **without** `cross_val_predict`: loop over `cv.split(X)`, clone and fit the model on each training fold, and fill an array with predictions for the held-out fold. Each row must be predicted exactly once.
2. Use it to produce `oof_extra` for ExtraTrees on `(Ah_train, yh_train)` with `kf_reg`.
3. Append `oof_extra` as a fifth column to `Z_oof_reg`, fit the non-negative meta-learner, and compute the **test RMSE of the 5-model stack** → `rmse_five`. (You'll need ExtraTrees' predictions on the test set from a model fitted on all training data.)

<details><summary>💡 Hint 1</summary>

`from sklearn.base import clone`. Inside the loop: `m = clone(model).fit(X[tr_idx], y[tr_idx])` then `oof[va_idx] = m.predict(X[va_idx])`.

</details>

<details><summary>💡 Hint 2</summary>

Start with `oof = np.zeros(len(y))`; `for tr_idx, va_idx in cv.split(X): ...`.

</details>

<details><summary>💡 Hint 3</summary>

`Z5_oof = Z_oof_reg.assign(ExtraTrees=oof_extra)`; `Z5_test = Z_test_reg.assign(ExtraTrees=et_full.predict(Ah_test))`; meta = `LinearRegression(positive=True).fit(Z5_oof, yh_train)`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
from sklearn.base import clone

def oof_predictions(model, X, y, cv):
    oof = np.zeros(len(y))
    # YOUR CODE HERE
    return oof

et = ExtraTreesRegressor(n_estimators=300, max_features=0.5, random_state=SEED, n_jobs=-1)
oof_extra = TODO     # YOUR CODE HERE
rmse_five = TODO     # YOUR CODE HERE

# ---- self-check (don't edit) ----
def _ok():
    ref = cross_val_predict(et, Ah_train, yh_train, cv=kf_reg)
    return (len(oof_extra) == len(yh_train) and np.corrcoef(oof_extra, ref)[0, 1] > 0.999 and 85 < rmse_five < 105)
check("05.4", _ok, "Every row must get exactly one out-of-fold prediction (it should match cross_val_predict); the 5-model stack's test RMSE should be about 96.",
      needs=[oof_extra, rmse_five])

In [16]:
scoreboard()

Scoreboard: 4 solved, 0 to fix, 0 not attempted
   ✅ 05.1
   ✅ 05.2
   ✅ 05.3
   ✅ 05.4


## 🏁 Recap

* **Hard voting** counts class votes; **soft voting** averages probabilities (usually better, and gives probabilities for AUC).
* Ensembles gain from **diversity**: measure the correlation of *errors*, and combine different families / target views.
* **Stacking** learns the combination with a meta-learner trained on **out-of-fold** predictions. In-sample predictions → **leakage** (we watched the naive stack hand the random forest a weight of ≈ 7.5 and collapse).
* Prefer simple meta-learners; **non-negative linear** weights are a robust default for regression.
* On housing, an equal-weight average of diverse models beat every member by ~10%; stacking added a little more.
* Stacking costs complexity and latency: adopt it when the accuracy gain is worth it.

### ➡️ Next up: **Lesson 06: Cross-Validation & Model Comparison**
We keep comparing models on a single split and declaring winners by a few RMSE points. How do we know those differences are *real*? That's the next lesson, with paired tests, nested CV, leakage traps, and time-aware validation.